# 28. 異常利益と混在データの調査
出典：FX (3).ipynb、元セルindex [59, 60, 61, 62]。保存出力は results/imported_fx3/。
研究履歴の原本です。Notebookの変数・価格CSV・学習済みファイルに依存します。
失敗した試行も保管しています。一括実行やAPI接続を開始する入口ではありません。
元コード内の指示・自動判定名は資料として保存しています。独立した検証済みの結論とは区別してください。


## 元セルindex 59
構文状態：valid


In [ ]:
# ============================================================
# FINAL BACKTEST AUDIT
# Leakage / Timing / Cost / Overlap / Outlier / PF Audit
#
# 特に2022・2025の異常に高いPFを監査する
#
# 必要:
#   bars
#   CHAMPION_REINTEGRATION_TRADES
#
# もし変数名が異なる場合はTRADESも自動で探す
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

AUDIT_COST = 0.00004

AUDIT_YEARS = [
    2022,
    2025,
]


# ============================================================
# 0. 表示補助
# ============================================================

def show_table(title, df):

    print()
    print("=" * 100)
    print(title)
    print("=" * 100)

    if df is None or len(df) == 0:
        print("No data")
        return

    try:
        display(df)
    except Exception:
        print(df.to_string())


# ============================================================
# 1. TRADESを取得
# ============================================================

trade_candidates = [
    "CHAMPION_REINTEGRATION_TRADES",
    "TRADES",
    "all_hgb_trades",
]

TRADE_SOURCE_NAME = None
T = None

for name in trade_candidates:

    obj = globals().get(name)

    if isinstance(obj, pd.DataFrame) and len(obj) > 0:

        T = obj.copy()
        TRADE_SOURCE_NAME = name
        break


if T is None:

    raise RuntimeError(
        "取引結果DataFrameが見つかりません。\n"
        "CHAMPION_REINTEGRATION_TESTを先に実行してください。"
    )


print("=" * 100)
print("BACKTEST AUDIT START")
print("=" * 100)

print(
    "Trade source:",
    TRADE_SOURCE_NAME
)

print(
    "Trade rows:",
    f"{len(T):,}"
)


# ============================================================
# 2. barsを確認
# ============================================================

if "bars" not in globals():

    raise RuntimeError(
        "`bars` がありません。\n"
        "15分足修復セルを先に実行してください。"
    )


if not isinstance(bars, pd.DataFrame):

    raise TypeError(
        "bars がDataFrameではありません。"
    )


B = bars.copy()

B.columns = [
    str(c).strip().lower()
    for c in B.columns
]


need_ohlc = [
    "open",
    "high",
    "low",
    "close"
]


missing = [
    c
    for c in need_ohlc
    if c not in B.columns
]


if missing:

    raise RuntimeError(
        f"barsにOHLC列が不足しています: {missing}"
    )


if not isinstance(
    B.index,
    pd.DatetimeIndex
):

    raise RuntimeError(
        "bars.index がDatetimeIndexではありません。"
    )


B.index = pd.to_datetime(
    B.index,
    utc=True,
    errors="coerce"
)


B = B.loc[
    ~B.index.isna()
].copy()


B = (
    B
    .sort_index()
    .loc[
        lambda x:
        ~x.index.duplicated(
            keep="first"
        )
    ]
)


for c in need_ohlc:

    B[c] = pd.to_numeric(
        B[c],
        errors="coerce"
    )


B = B.dropna(
    subset=need_ohlc
)


print(
    "Bars:",
    f"{len(B):,}"
)

print(
    "Bars period:",
    B.index.min(),
    "->",
    B.index.max()
)


# ============================================================
# 3. Trade indexをsignal_timeとして正規化
# ============================================================

if "signal_time" in T.columns:

    signal_index = pd.to_datetime(
        T["signal_time"],
        utc=True,
        errors="coerce"
    )

else:

    signal_index = pd.to_datetime(
        T.index,
        utc=True,
        errors="coerce"
    )


valid_signal = ~signal_index.isna()

T = T.loc[
    valid_signal
].copy()

T.index = pd.DatetimeIndex(
    signal_index[
        valid_signal
    ]
)

T.index.name = "signal_time"


# ============================================================
# 4. 必須列チェック
# ============================================================

required_trade_cols = [
    "feature_set",
    "test_year",
    "gross_return",
]


missing_trade_cols = [
    c
    for c in required_trade_cols
    if c not in T.columns
]


if missing_trade_cols:

    raise RuntimeError(
        "Trade DataFrameに必要列がありません: "
        f"{missing_trade_cols}"
    )


T["test_year"] = pd.to_numeric(
    T["test_year"],
    errors="coerce"
)


# position_sizeが無ければ1.0
if "position_size" not in T.columns:

    print(
        "[INFO] position_size列なし → 1.0として監査"
    )

    T["position_size"] = 1.0


T["position_size"] = pd.to_numeric(
    T["position_size"],
    errors="coerce"
)


T["gross_return"] = pd.to_numeric(
    T["gross_return"],
    errors="coerce"
)


# ------------------------------------------------------------
# 実際に使われたreturn列を探す
# ------------------------------------------------------------

RETURN_COLUMN = None

for c in [
    "net_return",
    "sized_return",
    "base_net_return",
]:

    if c in T.columns:

        RETURN_COLUMN = c
        break


if RETURN_COLUMN is None:

    print(
        "[INFO] return列なし → 監査用net_returnを作成"
    )

    T["audit_net_return"] = (
        T["position_size"]
        *
        (
            T["gross_return"]
            -
            AUDIT_COST
        )
    )

    RETURN_COLUMN = "audit_net_return"


T[RETURN_COLUMN] = pd.to_numeric(
    T[RETURN_COLUMN],
    errors="coerce"
)


print(
    "Return column:",
    RETURN_COLUMN
)


# ============================================================
# 5. 時刻情報を標準化
# ============================================================

if "entry_time" in T.columns:

    T["entry_time_audit"] = pd.to_datetime(
        T["entry_time"],
        utc=True,
        errors="coerce"
    )

else:

    T["entry_time_audit"] = (
        T.index
        +
        pd.Timedelta(
            minutes=15
        )
    )


if "label_end" in T.columns:

    T["label_end_audit"] = pd.to_datetime(
        T["label_end"],
        utc=True,
        errors="coerce"
    )

else:

    T["label_end_audit"] = (
        T.index
        +
        pd.Timedelta(
            minutes=45
        )
    )


# ============================================================
# 6. Timing Audit
#
# signal t
# entry = Open(t+1) → t + 15m
# exit  = Close(t+2)
#
# Close(t+2)のbar label = t+30m
# bar close時刻 = t+45m
#
# 実ポジション保持:
# entry t+15 → exit t+45 = 30分
# ============================================================

expected_entry_time = (
    T.index
    +
    pd.Timedelta(
        minutes=15
    )
)


expected_label_end = (
    T.index
    +
    pd.Timedelta(
        minutes=45
    )
)


entry_timing_bad = (
    T["entry_time_audit"]
    !=
    expected_entry_time
)


label_timing_bad = (
    T["label_end_audit"]
    !=
    expected_label_end
)


holding_minutes = (
    T["label_end_audit"]
    -
    T["entry_time_audit"]
).dt.total_seconds() / 60


holding_bad = ~np.isclose(
    holding_minutes,
    30.0,
    equal_nan=False
)


# ============================================================
# 7. barsからEntry / Exit Priceを再構築
# ============================================================

entry_bar_time = (
    T.index
    +
    pd.Timedelta(
        minutes=15
    )
)


exit_bar_time = (
    T.index
    +
    pd.Timedelta(
        minutes=30
    )
)


entry_price_calc = (
    B["open"]
    .reindex(
        entry_bar_time
    )
    .to_numpy()
)


exit_price_calc = (
    B["close"]
    .reindex(
        exit_bar_time
    )
    .to_numpy()
)


T["audit_entry_price"] = (
    entry_price_calc
)


T["audit_exit_price"] = (
    exit_price_calc
)


T["audit_future_return"] = (
    T["audit_exit_price"]
    /
    T["audit_entry_price"]
    -
    1
)


missing_prices = (
    T["audit_entry_price"].isna()
    |
    T["audit_exit_price"].isna()
)


# ============================================================
# 8. future_return再計算チェック
# ============================================================

if "future_return" in T.columns:

    T["future_return"] = pd.to_numeric(
        T["future_return"],
        errors="coerce"
    )

    future_return_diff = (
        T["future_return"]
        -
        T["audit_future_return"]
    ).abs()


    future_return_bad = (
        future_return_diff
        >
        1e-10
    )

else:

    future_return_bad = pd.Series(
        False,
        index=T.index
    )


# ============================================================
# 9. Gross Return方向チェック
# ============================================================

if "p_up" in T.columns:

    T["audit_side"] = np.where(
        T["p_up"] >= 0.5,
        "BUY",
        "SELL"
    )


    side_sign = np.where(
        T["p_up"] >= 0.5,
        1.0,
        -1.0
    )


    gross_expected = (
        T["audit_future_return"]
        *
        side_sign
    )


    gross_diff = (
        T["gross_return"]
        -
        gross_expected
    ).abs()


    gross_return_bad = (
        gross_diff
        >
        1e-10
    )

else:

    T["audit_side"] = "UNKNOWN"

    gross_return_bad = pd.Series(
        False,
        index=T.index
    )


# ============================================================
# 10. Cost / Position Sizing再計算
# ============================================================

if RETURN_COLUMN == "base_net_return":

    expected_net = (
        T["gross_return"]
        -
        AUDIT_COST
    )

else:

    expected_net = (
        T["position_size"]
        *
        (
            T["gross_return"]
            -
            AUDIT_COST
        )
    )


net_diff = (
    T[RETURN_COLUMN]
    -
    expected_net
).abs()


cost_bad = (
    net_diff
    >
    1e-10
)


size_bad = (
    ~np.isfinite(
        T["position_size"]
    )
    |
    (T["position_size"] < 0.25)
    |
    (T["position_size"] > 2.0)
)


# ============================================================
# 11. Year Boundary / Split Audit
# ============================================================

year_int = (
    T["test_year"]
    .round()
    .astype("Int64")
)


year_start = pd.to_datetime(
    year_int.astype(str)
    +
    "-01-01",
    utc=True,
    errors="coerce"
)


year_end = pd.to_datetime(
    (year_int + 1).astype(str)
    +
    "-01-01",
    utc=True,
    errors="coerce"
)


year_boundary_bad = (
    (T.index < year_start)
    |
    (T.index >= year_end)
    |
    (T["label_end_audit"] > year_end)
)


# ============================================================
# 12. Duplicate / Overlap Audit
# ============================================================

duplicate_rows = []

overlap_rows = []


for (
    feature_set,
    year
), g in T.groupby(
    [
        "feature_set",
        "test_year"
    ],
    dropna=False
):

    g = g.sort_values(
        "entry_time_audit"
    )


    duplicate_count = int(
        g.index.duplicated().sum()
    )


    previous_exit = (
        g["label_end_audit"]
        .shift(1)
    )


    overlap_mask = (
        g["entry_time_audit"]
        <
        previous_exit
    )


    overlap_count = int(
        overlap_mask.fillna(
            False
        ).sum()
    )


    duplicate_rows.append(
        {
            "feature_set":
                feature_set,

            "test_year":
                year,

            "trades":
                len(g),

            "duplicate_signal_count":
                duplicate_count,
        }
    )


    overlap_rows.append(
        {
            "feature_set":
                feature_set,

            "test_year":
                year,

            "trades":
                len(g),

            "overlap_count":
                overlap_count,
        }
    )


DUPLICATE_TABLE = pd.DataFrame(
    duplicate_rows
)


OVERLAP_TABLE = pd.DataFrame(
    overlap_rows
)


# ============================================================
# 13. PF / Performanceを完全再計算
# ============================================================

def calculate_stats(
    returns
):

    r = np.asarray(
        returns,
        dtype=float
    )

    r = r[
        np.isfinite(r)
    ]


    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": np.nan,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }


    gains = r[
        r > 0
    ].sum()


    losses = -r[
        r < 0
    ].sum()


    if losses > 0:

        pf = (
            gains /
            losses
        )

    elif gains > 0:

        pf = np.inf

    else:

        pf = np.nan


    equity = np.r_[
        1.0,
        np.cumprod(
            1 + r
        )
    ]


    peak = np.maximum.accumulate(
        equity
    )


    dd = (
        equity /
        peak -
        1
    )


    growth = (
        equity[-1] - 1
    )


    max_dd = (
        dd.min()
    )


    rdd = (
        growth /
        abs(max_dd)
        if max_dd < 0
        else np.nan
    )


    return {
        "trades":
            len(r),

        "win_rate":
            np.mean(
                r > 0
            ),

        "avg_return":
            np.mean(r),

        "profit_factor":
            pf,

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            rdd,
    }


yearly_rows = []


for (
    feature_set,
    year
), g in T.groupby(
    [
        "feature_set",
        "test_year"
    ]
):

    s = calculate_stats(
        g[RETURN_COLUMN]
    )


    yearly_rows.append(
        {
            "feature_set":
                feature_set,

            "test_year":
                int(year),

            **s
        }
    )


YEARLY_RECALC = pd.DataFrame(
    yearly_rows
).sort_values(
    [
        "feature_set",
        "test_year"
    ]
)


# ============================================================
# 14. 2022 / 2025 Outlier Concentration
# ============================================================

outlier_rows = []


for (
    feature_set,
    year
), g in T[
    T["test_year"].isin(
        AUDIT_YEARS
    )
].groupby(
    [
        "feature_set",
        "test_year"
    ]
):

    r = (
        g[RETURN_COLUMN]
        .dropna()
        .sort_values(
            ascending=False
        )
    )


    positive = r[
        r > 0
    ]


    total_positive = (
        positive.sum()
    )


    if total_positive > 0:

        top1_share = (
            positive.head(1).sum()
            /
            total_positive
        )


        top5_share = (
            positive.head(5).sum()
            /
            total_positive
        )


        top10_share = (
            positive.head(10).sum()
            /
            total_positive
        )

    else:

        top1_share = np.nan
        top5_share = np.nan
        top10_share = np.nan


    s = calculate_stats(
        r
    )


    outlier_rows.append(
        {
            "feature_set":
                feature_set,

            "year":
                int(year),

            "trades":
                s["trades"],

            "profit_factor":
                s["profit_factor"],

            "avg_return":
                s["avg_return"],

            "top1_profit_share":
                top1_share,

            "top5_profit_share":
                top5_share,

            "top10_profit_share":
                top10_share,

            "largest_trade":
                r.iloc[0]
                if len(r)
                else np.nan,

            "worst_trade":
                r.iloc[-1]
                if len(r)
                else np.nan,
        }
    )


OUTLIER_CONCENTRATION = (
    pd.DataFrame(
        outlier_rows
    )
)


# ============================================================
# 15. Top Winning / Losing Trades
# ============================================================

audit_subset = T[
    T["test_year"].isin(
        AUDIT_YEARS
    )
].copy()


top_columns = [
    "feature_set",
    "test_year",
    "audit_side",
    "confidence",
    "p_up",
    "position_size",
    "audit_entry_price",
    "audit_exit_price",
    "audit_future_return",
    "gross_return",
    RETURN_COLUMN,
]


top_columns = [
    c
    for c in top_columns
    if c in audit_subset.columns
]


TOP_WINNERS = (
    audit_subset
    .sort_values(
        RETURN_COLUMN,
        ascending=False
    )
    .head(20)[
        top_columns
    ]
)


TOP_LOSERS = (
    audit_subset
    .sort_values(
        RETURN_COLUMN,
        ascending=True
    )
    .head(20)[
        top_columns
    ]
)


# ============================================================
# 16. Session / Direction Concentration
# ============================================================

audit_subset[
    "utc_hour"
] = audit_subset.index.hour


SESSION_CONCENTRATION = (

    audit_subset

    .groupby(
        [
            "feature_set",
            "test_year",
            "utc_hour"
        ]
    )

    .size()

    .rename(
        "trades"
    )

    .reset_index()

    .sort_values(
        [
            "feature_set",
            "test_year",
            "trades"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)


SIDE_CONCENTRATION = (

    audit_subset

    .groupby(
        [
            "feature_set",
            "test_year",
            "audit_side"
        ]
    )

    .size()

    .rename(
        "trades"
    )

    .reset_index()
)


# ============================================================
# 17. 大きすぎる30分returnを検査
# ============================================================

ABS_RETURN_WARNING_LEVEL = 0.02
# 30分で ±2%以上なら要確認
# 自動FAILではなくWARNING


EXTREME_PRICE_MOVES = T.loc[
    T[
        "audit_future_return"
    ].abs()
    >
    ABS_RETURN_WARNING_LEVEL
].copy()


extreme_cols = [
    "feature_set",
    "test_year",
    "audit_entry_price",
    "audit_exit_price",
    "audit_future_return",
    "gross_return",
    "position_size",
    RETURN_COLUMN,
]


extreme_cols = [
    c
    for c in extreme_cols
    if c in EXTREME_PRICE_MOVES.columns
]


EXTREME_PRICE_MOVES = (
    EXTREME_PRICE_MOVES[
        extreme_cols
    ]
    .sort_values(
        "audit_future_return",
        key=lambda s: s.abs(),
        ascending=False
    )
)


# ============================================================
# 18. Feature Name Leakage Check
# ============================================================

feature_list_names = [
    "BASE_FEATURES",
    "CHAMPION_FEATURES",
    "REGIME_ADDITIONS",
    "VOLATILITY_ADDITIONS",
]


danger_tokens = [
    "future",
    "target",
    "label",
    "entry_price",
    "exit_price",
    "probability",
    "confidence",
    "p_up",
]


feature_leak_rows = []


for list_name in feature_list_names:

    feature_list = globals().get(
        list_name
    )


    if not isinstance(
        feature_list,
        (
            list,
            tuple,
            pd.Index,
            np.ndarray,
        )
    ):
        continue


    for feature in feature_list:

        f = str(
            feature
        ).lower()


        matched = [
            token
            for token in danger_tokens
            if token in f
        ]


        if matched:

            feature_leak_rows.append(
                {
                    "feature_list":
                        list_name,

                    "feature":
                        feature,

                    "matched_tokens":
                        ", ".join(
                            matched
                        )
                }
            )


FEATURE_NAME_LEAK_CHECK = (
    pd.DataFrame(
        feature_leak_rows
    )
)


# ============================================================
# 19. Hard Audit Summary
# ============================================================

AUDIT_CHECKS = pd.DataFrame(
    [
        {
            "check":
                "Missing reconstructed prices",

            "fail_count":
                int(
                    missing_prices.sum()
                ),
        },

        {
            "check":
                "Entry timing mismatch",

            "fail_count":
                int(
                    entry_timing_bad.sum()
                ),
        },

        {
            "check":
                "Exit/label timing mismatch",

            "fail_count":
                int(
                    label_timing_bad.sum()
                ),
        },

        {
            "check":
                "Holding period != 30m",

            "fail_count":
                int(
                    holding_bad.sum()
                ),
        },

        {
            "check":
                "Future return mismatch",

            "fail_count":
                int(
                    future_return_bad.sum()
                ),
        },

        {
            "check":
                "Gross return mismatch",

            "fail_count":
                int(
                    gross_return_bad.sum()
                ),
        },

        {
            "check":
                "Cost / sizing mismatch",

            "fail_count":
                int(
                    cost_bad.sum()
                ),
        },

        {
            "check":
                "Position size outside 0.25-2.0",

            "fail_count":
                int(
                    size_bad.sum()
                ),
        },

        {
            "check":
                "Year / split boundary mismatch",

            "fail_count":
                int(
                    year_boundary_bad.sum()
                ),
        },

        {
            "check":
                "Duplicate signals",

            "fail_count":
                int(
                    DUPLICATE_TABLE[
                        "duplicate_signal_count"
                    ].sum()
                ),
        },

        {
            "check":
                "Overlapping positions",

            "fail_count":
                int(
                    OVERLAP_TABLE[
                        "overlap_count"
                    ].sum()
                ),
        },

        {
            "check":
                "Suspicious feature names",

            "fail_count":
                int(
                    len(
                        FEATURE_NAME_LEAK_CHECK
                    )
                ),
        },
    ]
)


TOTAL_HARD_FAILURES = int(
    AUDIT_CHECKS[
        "fail_count"
    ].sum()
)


# ============================================================
# 20. Warning判定
# ============================================================

warnings_list = []


# PF 5超
high_pf = YEARLY_RECALC[
    YEARLY_RECALC[
        "profit_factor"
    ] > 5
]


if len(high_pf):

    warnings_list.append(
        f"PF > 5 の年が {len(high_pf)} 件あります。"
    )


# Top10が利益の50%以上
if len(
    OUTLIER_CONCENTRATION
):

    concentrated = (
        OUTLIER_CONCENTRATION[
            "top10_profit_share"
        ] > 0.50
    )

    if concentrated.any():

        warnings_list.append(
            "上位10トレードが総利益の50%以上を"
            "占める年があります。"
        )


# 大きい価格変動
if len(
    EXTREME_PRICE_MOVES
):

    warnings_list.append(
        f"30分で±2%以上の価格変動が "
        f"{len(EXTREME_PRICE_MOVES)} 件あります。"
    )


# ============================================================
# 21. Final Audit Decision
# ============================================================

if TOTAL_HARD_FAILURES == 0:

    AUDIT_DECISION = (
        "HARD AUDIT PASSED"
    )

else:

    AUDIT_DECISION = (
        "HARD AUDIT FAILED"
    )


# ============================================================
# 22. 結果表示
# ============================================================

show_table(
    "HARD AUDIT CHECKS",
    AUDIT_CHECKS
)


show_table(
    "RECALCULATED YEARLY PERFORMANCE",
    YEARLY_RECALC
)


show_table(
    "2022 / 2025 OUTLIER CONCENTRATION",
    OUTLIER_CONCENTRATION
)


show_table(
    "DUPLICATE SIGNAL CHECK",
    DUPLICATE_TABLE
)


show_table(
    "OVERLAPPING POSITION CHECK",
    OVERLAP_TABLE
)


show_table(
    "TOP 20 WINNING TRADES — 2022 / 2025",
    TOP_WINNERS
)


show_table(
    "TOP 20 LOSING TRADES — 2022 / 2025",
    TOP_LOSERS
)


show_table(
    "2022 / 2025 SIDE CONCENTRATION",
    SIDE_CONCENTRATION
)


show_table(
    "2022 / 2025 UTC HOUR CONCENTRATION",
    SESSION_CONCENTRATION
)


show_table(
    "EXTREME 30-MIN PRICE MOVES",
    EXTREME_PRICE_MOVES
)


show_table(
    "FEATURE NAME LEAK CHECK",
    FEATURE_NAME_LEAK_CHECK
)


# ============================================================
# 23. 最後の診断
# ============================================================

print()
print("=" * 100)
print("FINAL AUDIT DIAGNOSIS")
print("=" * 100)

print(
    "Trade source:",
    TRADE_SOURCE_NAME
)

print(
    "Return column:",
    RETURN_COLUMN
)

print(
    "Trades audited:",
    f"{len(T):,}"
)

print(
    "Hard failures:",
    TOTAL_HARD_FAILURES
)

print(
    "Decision:",
    AUDIT_DECISION
)


if warnings_list:

    print()
    print("WARNINGS:")

    for w in warnings_list:

        print(
            " -",
            w
        )

else:

    print()
    print(
        "Warnings: none"
    )


print()


if AUDIT_DECISION == "HARD AUDIT PASSED":

    print(
        "次の段階:"
    )

    print(
        "BASE vs BASE+REGIME vs BASE+VOL+REGIME"
    )

    print(
        "のFINAL CANDIDATE TOURNAMENTへ進めます。"
    )

else:

    print(
        "次のFeature/Model検証へ進まないでください。"
    )

    print(
        "fail_count > 0 の原因を修正して、"
        "バックテストを再計算する必要があります。"
    )


# ============================================================
# 24. 次回用Notebook変数
# ============================================================

FX_AUDIT_TRADES = T.copy()

FX_AUDIT_CHECKS = AUDIT_CHECKS.copy()

FX_AUDIT_YEARLY = YEARLY_RECALC.copy()

FX_AUDIT_OUTLIERS = OUTLIER_CONCENTRATION.copy()

FX_AUDIT_TOP_WINNERS = TOP_WINNERS.copy()

FX_AUDIT_TOP_LOSERS = TOP_LOSERS.copy()

FX_AUDIT_EXTREME_MOVES = EXTREME_PRICE_MOVES.copy()

FX_AUDIT_DECISION = AUDIT_DECISION


print()
print("=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

print(
    "FX_AUDIT_CHECKS"
)

print(
    "FX_AUDIT_YEARLY"
)

print(
    "FX_AUDIT_OUTLIERS"
)

print(
    "FX_AUDIT_TOP_WINNERS"
)

print(
    "FX_AUDIT_TOP_LOSERS"
)

print(
    "FX_AUDIT_DECISION"
)


## 元セルindex 60
構文状態：valid


In [ ]:
# ============================================================
# MARKET DATA INTEGRITY AUDIT + SAFE 15M RECONSTRUCTION
#
# 目的:
#   1. resample前の元データを探す
#   2. 元から00/15/30/45分に存在する行だけを正式15分足候補にする
#   3. 05/10/20/25...分の行は絶対に15分足へ混ぜない
#   4. 現在のbarsに発生した158→143などの人工jumpを検査
#   5. 安全なら bars_clean_candidate を作る
#
# 重要:
#   このセルはまだ `bars` を上書きしません。
# ============================================================

import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 200)

OUTPUT_DIR = "data_integrity_audit"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 0. 補助
# ============================================================

def show(title, df, max_rows=50):

    print()
    print("=" * 110)
    print(title)
    print("=" * 110)

    if df is None or len(df) == 0:
        print("No data")
        return

    try:
        display(df.head(max_rows))
    except Exception:
        print(df.head(max_rows).to_string())


def normalize_ohlc(frame, name="data"):

    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{name} is not DataFrame")

    x = frame.copy()

    x.columns = [
        str(c).strip().lower()
        for c in x.columns
    ]

    required = [
        "open",
        "high",
        "low",
        "close",
    ]

    missing = [
        c
        for c in required
        if c not in x.columns
    ]

    if missing:
        raise ValueError(
            f"{name}: OHLC columns missing: {missing}"
        )

    # ----------------------------
    # timestamp
    # ----------------------------

    if not isinstance(
        x.index,
        pd.DatetimeIndex
    ):

        time_col = next(
            (
                c
                for c in [
                    "timestamp",
                    "datetime",
                    "date",
                    "time",
                ]
                if c in x.columns
            ),
            None
        )

        if time_col is None:
            raise ValueError(
                f"{name}: datetime index/column not found"
            )

        x.index = pd.to_datetime(
            x.pop(time_col),
            utc=True,
            errors="coerce"
        )

    else:

        x.index = pd.to_datetime(
            x.index,
            utc=True,
            errors="coerce"
        )

    x = x.loc[
        ~x.index.isna()
    ].copy()

    x = x.sort_index()

    x = x[
        required
    ].copy()

    for c in required:

        x[c] = pd.to_numeric(
            x[c],
            errors="coerce"
        )

    x = x.dropna()

    return x


# ============================================================
# 1. 現在のbars
# ============================================================

if "bars" not in globals():
    raise RuntimeError(
        "`bars` がNotebookにありません。"
    )

CURRENT = normalize_ohlc(
    bars,
    "current bars"
)

print("=" * 110)
print("CURRENT BARS")
print("=" * 110)

print(
    "Rows:",
    f"{len(CURRENT):,}"
)

print(
    "Period:",
    CURRENT.index.min(),
    "->",
    CURRENT.index.max()
)


# ============================================================
# 2. resample前の元DataFrameを探す
#
# 最優先は bars_raw
# ============================================================

candidate_names = [

    "bars_raw",
    "BARS_RAW",
    "raw_bars",
    "bars_original",
    "original_bars",
    "raw_data",
    "df_raw",
]


RAW_NAME = None
RAW = None


# ------------------------------------------------------------
# 明示候補
# ------------------------------------------------------------

for name in candidate_names:

    obj = globals().get(name)

    if not isinstance(
        obj,
        pd.DataFrame
    ):
        continue

    try:

        temp = normalize_ohlc(
            obj,
            name
        )

    except Exception:

        continue


    # 現在より行数が多いものを優先
    if len(temp) > len(CURRENT):

        RAW_NAME = name
        RAW = temp
        break


# ------------------------------------------------------------
# 見つからない場合Notebook namespace全検索
# ------------------------------------------------------------

if RAW is None:

    possible = []

    for name, obj in list(globals().items()):

        if not isinstance(
            obj,
            pd.DataFrame
        ):
            continue

        if obj is bars:
            continue

        try:

            temp = normalize_ohlc(
                obj,
                name
            )

        except Exception:

            continue

        if len(temp) <= len(CURRENT):
            continue

        possible.append(
            (
                len(temp),
                name,
                temp
            )
        )


    if possible:

        possible.sort(
            reverse=True,
            key=lambda x: x[0]
        )

        _, RAW_NAME, RAW = possible[0]


if RAW is None:

    raise RuntimeError(
        "\nresample前の元データをNotebook内から発見できませんでした。\n"
        "\n安全のため、現在の266,510行barsから元データを逆算することはしません。"
        "\nJupyter Kernelを再起動せずに作業しているなら `bars_raw` が残っている可能性があります。"
        "\nもし無ければ元CSVを再読み込みしてください。"
    )


print()
print("=" * 110)
print("RAW SOURCE FOUND")
print("=" * 110)

print(
    "Variable:",
    RAW_NAME
)

print(
    "Rows:",
    f"{len(RAW):,}"
)

print(
    "Period:",
    RAW.index.min(),
    "->",
    RAW.index.max()
)


# ============================================================
# 3. 元データを
#
# 正式15分grid
# vs
# off-grid
#
# に完全分離
# ============================================================

exact_grid_mask = (

    (RAW.index.minute % 15 == 0)

    &

    (RAW.index.second == 0)

    &

    (RAW.index.microsecond == 0)
)


EXACT_RAW = RAW.loc[
    exact_grid_mask
].copy()


OFFGRID_RAW = RAW.loc[
    ~exact_grid_mask
].copy()


print()
print("=" * 110)
print("RAW GRID DECOMPOSITION")
print("=" * 110)

print(
    "Raw rows:",
    f"{len(RAW):,}"
)

print(
    "Exact 15m rows:",
    f"{len(EXACT_RAW):,}"
)

print(
    "Off-grid rows:",
    f"{len(OFFGRID_RAW):,}"
)

print(
    "Off-grid ratio:",
    f"{len(OFFGRID_RAW)/len(RAW)*100:.3f}%"
)


# ============================================================
# 4. Exact-grid timestamp重複を監査
# ============================================================

duplicate_mask = EXACT_RAW.index.duplicated(
    keep=False
)

duplicate_exact = EXACT_RAW.loc[
    duplicate_mask
].copy()


conflict_rows = []


if len(duplicate_exact):

    for ts, group in duplicate_exact.groupby(
        level=0
    ):

        unique = group.drop_duplicates()

        if len(unique) > 1:

            row = {
                "timestamp": ts,
                "rows": len(group),
                "unique_ohlc_rows": len(unique),
            }

            for c in [
                "open",
                "high",
                "low",
                "close"
            ]:

                row[
                    f"{c}_min"
                ] = group[c].min()

                row[
                    f"{c}_max"
                ] = group[c].max()

            conflict_rows.append(
                row
            )


EXACT_DUPLICATE_CONFLICTS = pd.DataFrame(
    conflict_rows
)


show(
    "CONFLICTING EXACT-GRID DUPLICATES",
    EXACT_DUPLICATE_CONFLICTS
)


if len(
    EXACT_DUPLICATE_CONFLICTS
) == 0:

    # 完全一致duplicateだけなら1つにする
    CLEAN = EXACT_RAW.loc[
        ~EXACT_RAW.index.duplicated(
            keep="first"
        )
    ].copy()

else:

    CLEAN = EXACT_RAW.copy()


# ============================================================
# 5. OHLC整合性
# ============================================================

invalid_high = (

    CLEAN["high"]

    <

    CLEAN[
        [
            "open",
            "close",
            "low"
        ]
    ].max(axis=1)
)


invalid_low = (

    CLEAN["low"]

    >

    CLEAN[
        [
            "open",
            "close",
            "high"
        ]
    ].min(axis=1)
)


nonpositive = (

    CLEAN[
        [
            "open",
            "high",
            "low",
            "close"
        ]
    ]

    <= 0

).any(axis=1)


OHLC_BAD = CLEAN.loc[
    invalid_high
    |
    invalid_low
    |
    nonpositive
].copy()


show(
    "INVALID OHLC ROWS",
    OHLC_BAD
)


# ============================================================
# 6. 価格jump診断
#
# weekend等を混ぜないため、
# 前barから正確に15分連続している場合のみ評価
# ============================================================

def market_jump_table(
    frame,
    label
):

    x = frame.copy()

    prev_close = (
        x["close"]
        .shift(1)
    )

    previous_time = pd.Series(
        x.index,
        index=x.index
    ).shift(1)

    contiguous = (

        (
            pd.Series(
                x.index,
                index=x.index
            )
            -
            previous_time
        )

        ==

        pd.Timedelta(
            minutes=15
        )
    )


    x[
        "close_to_close_return"
    ] = np.where(

        contiguous,

        x["close"]
        /
        prev_close
        -
        1,

        np.nan
    )


    x[
        "open_gap_return"
    ] = np.where(

        contiguous,

        x["open"]
        /
        prev_close
        -
        1,

        np.nan
    )


    x[
        "bar_return"
    ] = (

        x["close"]
        /
        x["open"]
        -
        1
    )


    x[
        "intrabar_range"
    ] = (

        x["high"]
        /
        x["low"]
        -
        1
    )


    x[
        "source"
    ] = label


    severity = pd.concat(
        [
            x[
                "close_to_close_return"
            ].abs(),

            x[
                "open_gap_return"
            ].abs(),

            x[
                "bar_return"
            ].abs(),

            x[
                "intrabar_range"
            ].abs(),
        ],
        axis=1
    ).max(
        axis=1
    )


    x[
        "max_abs_move"
    ] = severity


    return x


CURRENT_DIAG = market_jump_table(
    CURRENT,
    "CURRENT_RESAMPLED"
)


CLEAN_DIAG = market_jump_table(
    CLEAN,
    "EXACT_GRID_ONLY"
)


# ------------------------------------------------------------
# 2%以上
# ------------------------------------------------------------

CURRENT_EXTREME = CURRENT_DIAG.loc[
    CURRENT_DIAG[
        "max_abs_move"
    ] > 0.02
].sort_values(
    "max_abs_move",
    ascending=False
)


CLEAN_EXTREME = CLEAN_DIAG.loc[
    CLEAN_DIAG[
        "max_abs_move"
    ] > 0.02
].sort_values(
    "max_abs_move",
    ascending=False
)


# ------------------------------------------------------------
# 5%以上 = catastrophic
# ------------------------------------------------------------

CURRENT_CATASTROPHIC = CURRENT_DIAG.loc[
    CURRENT_DIAG[
        "max_abs_move"
    ] > 0.05
].sort_values(
    "max_abs_move",
    ascending=False
)


CLEAN_CATASTROPHIC = CLEAN_DIAG.loc[
    CLEAN_DIAG[
        "max_abs_move"
    ] > 0.05
].sort_values(
    "max_abs_move",
    ascending=False
)


show(
    "CURRENT BARS — MOVES > 2%",
    CURRENT_EXTREME
)


show(
    "EXACT-GRID CLEAN CANDIDATE — MOVES > 2%",
    CLEAN_EXTREME
)


show(
    "CURRENT BARS — MOVES > 5%",
    CURRENT_CATASTROPHIC
)


show(
    "EXACT-GRID CLEAN CANDIDATE — MOVES > 5%",
    CLEAN_CATASTROPHIC
)


# ============================================================
# 7. Off-grid行が同じ15分bucketの正式価格と
#    どれくらい違うか確認
#
# ここが最重要
# ============================================================

if len(OFFGRID_RAW):

    OFFGRID_COMPARE = OFFGRID_RAW.copy()

    OFFGRID_COMPARE[
        "canonical_bucket"
    ] = OFFGRID_COMPARE.index.floor(
        "15min"
    )


    canonical_close_map = (
        CLEAN["close"]
    )


    OFFGRID_COMPARE[
        "canonical_close"
    ] = canonical_close_map.reindex(

        OFFGRID_COMPARE[
            "canonical_bucket"
        ]

    ).to_numpy()


    OFFGRID_COMPARE[
        "close_difference_pct"
    ] = (

        OFFGRID_COMPARE[
            "close"
        ]

        /

        OFFGRID_COMPARE[
            "canonical_close"
        ]

        -

        1
    )


    OFFGRID_COMPARE[
        "abs_close_difference_pct"
    ] = OFFGRID_COMPARE[
        "close_difference_pct"
    ].abs()


    OFFGRID_COMPARE = OFFGRID_COMPARE.sort_values(
        "abs_close_difference_pct",
        ascending=False
    )

else:

    OFFGRID_COMPARE = pd.DataFrame()


show(
    "OFF-GRID ROWS vs CANONICAL 15M PRICE — LARGEST DIFFERENCES",
    OFFGRID_COMPARE,
    max_rows=50
)


# ============================================================
# 8. 現在のresampled barsと
#    Exact-grid candidateを直接比較
# ============================================================

common = (
    CURRENT.index
    .intersection(
        CLEAN.index
    )
)


comparison = pd.DataFrame(
    index=common
)


for c in [
    "open",
    "high",
    "low",
    "close",
]:

    comparison[
        f"current_{c}"
    ] = CURRENT.loc[
        common,
        c
    ]

    comparison[
        f"clean_{c}"
    ] = CLEAN.loc[
        common,
        c
    ]

    comparison[
        f"{c}_abs_diff"
    ] = (

        comparison[
            f"current_{c}"
        ]

        -

        comparison[
            f"clean_{c}"
        ]

    ).abs()


comparison[
    "max_price_diff"
] = comparison[
    [
        "open_abs_diff",
        "high_abs_diff",
        "low_abs_diff",
        "close_abs_diff",
    ]
].max(axis=1)


CHANGED_BARS = comparison.loc[
    comparison[
        "max_price_diff"
    ] > 1e-10
].sort_values(
    "max_price_diff",
    ascending=False
)


show(
    "BARS CHANGED BY PREVIOUS RESAMPLE",
    CHANGED_BARS,
    max_rows=50
)


print()
print(
    "Bars changed by previous resample:",
    f"{len(CHANGED_BARS):,}"
)


# ============================================================
# 9. 特に問題だった日付を見る
# ============================================================

focus_ranges = [

    (
        "2025-01-05",
        "2025-01-11"
    ),

    (
        "2022-10-21",
        "2022-10-25"
    ),
]


focus_frames = []


for start, end in focus_ranges:

    clean_part = CLEAN_DIAG.loc[
        start:end
    ].copy()

    clean_part[
        "dataset"
    ] = "CLEAN"


    current_part = CURRENT_DIAG.loc[
        start:end
    ].copy()

    current_part[
        "dataset"
    ] = "CURRENT"


    focus_frames.append(
        clean_part
    )

    focus_frames.append(
        current_part
    )


FOCUS_WINDOWS = pd.concat(
    focus_frames
).sort_index()


focus_cols = [
    "dataset",
    "open",
    "high",
    "low",
    "close",
    "close_to_close_return",
    "open_gap_return",
    "bar_return",
    "intrabar_range",
    "max_abs_move",
]


show(
    "FOCUS WINDOWS — 2025 JAN / 2022 OCT",
    FOCUS_WINDOWS[
        focus_cols
    ],
    max_rows=150
)


# ============================================================
# 10. Summary
# ============================================================

summary = pd.DataFrame(
    [
        {
            "metric":
                "RAW rows",

            "value":
                len(RAW),
        },

        {
            "metric":
                "Exact-grid rows",

            "value":
                len(EXACT_RAW),
        },

        {
            "metric":
                "Off-grid rows",

            "value":
                len(OFFGRID_RAW),
        },

        {
            "metric":
                "Clean candidate rows",

            "value":
                len(CLEAN),
        },

        {
            "metric":
                "Exact-grid duplicate conflicts",

            "value":
                len(
                    EXACT_DUPLICATE_CONFLICTS
                ),
        },

        {
            "metric":
                "OHLC invalid rows",

            "value":
                len(
                    OHLC_BAD
                ),
        },

        {
            "metric":
                "Current moves >2%",

            "value":
                len(
                    CURRENT_EXTREME
                ),
        },

        {
            "metric":
                "Clean moves >2%",

            "value":
                len(
                    CLEAN_EXTREME
                ),
        },

        {
            "metric":
                "Current moves >5%",

            "value":
                len(
                    CURRENT_CATASTROPHIC
                ),
        },

        {
            "metric":
                "Clean moves >5%",

            "value":
                len(
                    CLEAN_CATASTROPHIC
                ),
        },

        {
            "metric":
                "Bars modified by prior resample",

            "value":
                len(
                    CHANGED_BARS
                ),
        },
    ]
)


show(
    "DATA INTEGRITY SUMMARY",
    summary
)


# ============================================================
# 11. Decision
# ============================================================

hard_problem = False

reasons = []


if len(
    EXACT_DUPLICATE_CONFLICTS
) > 0:

    hard_problem = True

    reasons.append(
        "Exact 15m timestamps contain conflicting duplicate OHLC rows."
    )


if len(
    OHLC_BAD
) > 0:

    hard_problem = True

    reasons.append(
        "Exact-grid data contains invalid OHLC rows."
    )


# 5%以上の15分moveがcleanにも残るなら
# 自動的に信用しない
if len(
    CLEAN_CATASTROPHIC
) > 0:

    hard_problem = True

    reasons.append(
        "Exact-grid candidate still contains >5% 15-minute price moves."
    )


if hard_problem:

    DATA_INTEGRITY_DECISION = (
        "MANUAL_REVIEW_REQUIRED"
    )

else:

    DATA_INTEGRITY_DECISION = (
        "CLEAN_GRID_CANDIDATE_READY"
    )


print()
print("=" * 110)
print("FINAL DATA INTEGRITY DECISION")
print("=" * 110)

print(
    "Raw source:",
    RAW_NAME
)

print(
    "Decision:",
    DATA_INTEGRITY_DECISION
)


if reasons:

    print()

    for reason in reasons:

        print(
            "-",
            reason
        )


if DATA_INTEGRITY_DECISION == (
    "CLEAN_GRID_CANDIDATE_READY"
):

    print()
    print(
        "Exact-grid only data passed the hard checks."
    )

    print(
        "Do NOT use the previous resampled bars."
    )

    print(
        "Next step will be to replace bars with bars_clean_candidate"
        " and rerun all final backtests."
    )

else:

    print()
    print(
        "Do not rerun the strategy yet."
    )

    print(
        "The original source itself still contains suspicious data"
        " and needs further diagnosis."
    )


# ============================================================
# 12. 重要変数を保存
# ============================================================

bars_clean_candidate = CLEAN.copy()

bars_offgrid_quarantine = OFFGRID_RAW.copy()

DATA_INTEGRITY_SUMMARY = summary.copy()

DATA_INTEGRITY_CHANGED_BARS = CHANGED_BARS.copy()

DATA_INTEGRITY_OFFGRID_COMPARE = OFFGRID_COMPARE.copy()

DATA_INTEGRITY_CLEAN_EXTREME = CLEAN_EXTREME.copy()

DATA_INTEGRITY_CURRENT_EXTREME = CURRENT_EXTREME.copy()

DATA_INTEGRITY_DECISION_RESULT = (
    DATA_INTEGRITY_DECISION
)


# ============================================================
# 13. CSVも保存
# ============================================================

bars_clean_candidate.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bars_clean_candidate_exact_15m.csv"
    ),
    index_label="timestamp"
)


bars_offgrid_quarantine.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "offgrid_quarantine.csv"
    ),
    index_label="timestamp"
)


CHANGED_BARS.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "bars_changed_by_previous_resample.csv"
    ),
    index_label="timestamp"
)


OFFGRID_COMPARE.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "offgrid_vs_canonical.csv"
    ),
    index_label="timestamp"
)


CLEAN_EXTREME.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "clean_extreme_moves.csv"
    ),
    index_label="timestamp"
)


summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "data_integrity_summary.csv"
    ),
    index=False
)


print()
print("=" * 110)
print("DATA INTEGRITY AUDIT COMPLETE")
print("=" * 110)

print(
    "Created:"
)

print(
    "bars_clean_candidate"
)

print(
    "bars_offgrid_quarantine"
)

print(
    "DATA_INTEGRITY_SUMMARY"
)

print(
    "DATA_INTEGRITY_CHANGED_BARS"
)

print(
    "DATA_INTEGRITY_OFFGRID_COMPARE"
)

print(
    "DATA_INTEGRITY_CLEAN_EXTREME"
)

print()
print(
    "Saved folder:",
    OUTPUT_DIR
)


## 元セルindex 61
構文状態：valid


In [ ]:
# ============================================================
# SOURCE CONTAMINATION AUDIT
# USDJPY 15m raw-data provenance / alternating-series diagnosis
#
# 目的:
# 1. bars_raw exact-grid内の異常gapを特定
# 2. 「142円系列 ↔ 158円系列」の交互混入を確認
# 3. 異常が集中している日・期間を特定
# 4. Notebook周辺の元CSV候補を探索
#
# barsは変更しない
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 300)

OUTPUT_DIR = Path("source_contamination_audit")
OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 0. raw source取得
# ============================================================

if "bars_raw" not in globals():

    raise RuntimeError(
        "`bars_raw` がありません。"
        "元275,704行DataFrameが必要です。"
    )


RAW = bars_raw.copy()

RAW.columns = [
    str(c).strip().lower()
    for c in RAW.columns
]


required = [
    "open",
    "high",
    "low",
    "close",
]


missing = [
    c
    for c in required
    if c not in RAW.columns
]


if missing:

    raise RuntimeError(
        f"OHLC列不足: {missing}"
    )


if not isinstance(
    RAW.index,
    pd.DatetimeIndex
):

    time_col = next(
        (
            c
            for c in [
                "timestamp",
                "datetime",
                "date",
                "time",
            ]
            if c in RAW.columns
        ),
        None
    )

    if time_col is None:

        raise RuntimeError(
            "timestampが見つかりません。"
        )

    RAW.index = pd.to_datetime(
        RAW.pop(time_col),
        utc=True,
        errors="coerce"
    )

else:

    RAW.index = pd.to_datetime(
        RAW.index,
        utc=True,
        errors="coerce"
    )


RAW = RAW.loc[
    ~RAW.index.isna()
].copy()


RAW = RAW.sort_index()


for c in required:

    RAW[c] = pd.to_numeric(
        RAW[c],
        errors="coerce"
    )


RAW = RAW.dropna(
    subset=required
)


print("=" * 100)
print("RAW SOURCE")
print("=" * 100)

print(
    "Rows:",
    f"{len(RAW):,}"
)

print(
    "Period:",
    RAW.index.min(),
    "->",
    RAW.index.max()
)


# ============================================================
# 1. exact 15mだけ
# ============================================================

grid_mask = (
    (RAW.index.minute % 15 == 0)
    &
    (RAW.index.second == 0)
    &
    (RAW.index.microsecond == 0)
)


X = RAW.loc[
    grid_mask
].copy()


X = X.loc[
    ~X.index.duplicated(
        keep="first"
    )
].copy()


print(
    "Exact-grid rows:",
    f"{len(X):,}"
)


# ============================================================
# 2. 前後価格
# ============================================================

X["prev_time"] = pd.Series(
    X.index,
    index=X.index
).shift(1)


X["next_time"] = pd.Series(
    X.index,
    index=X.index
).shift(-1)


X["prev_close"] = (
    X["close"].shift(1)
)


X["next_open"] = (
    X["open"].shift(-1)
)


X["next_close"] = (
    X["close"].shift(-1)
)


X["contiguous_prev"] = (

    (
        pd.Series(
            X.index,
            index=X.index
        )
        -
        X["prev_time"]
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


X["contiguous_next"] = (

    (
        X["next_time"]
        -
        pd.Series(
            X.index,
            index=X.index
        )
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


# ============================================================
# 3. gap / intrabar return
# ============================================================

X["open_gap"] = np.where(

    X["contiguous_prev"],

    X["open"]
    /
    X["prev_close"]
    -
    1,

    np.nan
)


X["close_gap"] = np.where(

    X["contiguous_prev"],

    X["close"]
    /
    X["prev_close"]
    -
    1,

    np.nan
)


X["bar_return"] = (
    X["close"]
    /
    X["open"]
    -
    1
)


X["intrabar_range"] = (
    X["high"]
    /
    X["low"]
    -
    1
)


# ============================================================
# 4. extreme gap
# ============================================================

GAP_THRESHOLD = 0.05


X["extreme_gap"] = (
    X["open_gap"].abs()
    >
    GAP_THRESHOLD
)


EXTREME = X.loc[
    X["extreme_gap"]
].copy()


print()
print("=" * 100)
print("EXTREME GAP COUNT")
print("=" * 100)

print(
    ">5% gaps:",
    len(EXTREME)
)


# ============================================================
# 5. 直後に逆方向へ戻るか
#
# contaminationの場合:
#
# 142 → 158 (+11%)
# 158 → 142 (-10%)
#
# のような往復が頻発するはず
# ============================================================

X["next_gap"] = (
    X["open_gap"]
    .shift(-1)
)


same_continuity = (
    X["contiguous_next"]
)


X["immediate_reversal"] = (

    X["extreme_gap"]

    &

    (
        X["next_gap"].abs()
        >
        GAP_THRESHOLD
    )

    &

    (
        np.sign(
            X["open_gap"]
        )

        !=

        np.sign(
            X["next_gap"]
        )
    )

    &

    same_continuity
)


REVERSALS = X.loc[
    X["immediate_reversal"]
].copy()


print(
    "Immediate opposite >5% reversals:",
    len(REVERSALS)
)


# ============================================================
# 6. suspicious bar window
# ============================================================

window_indices = set()


extreme_positions = np.flatnonzero(
    X["extreme_gap"].to_numpy()
)


for pos in extreme_positions:

    for p in range(
        max(
            0,
            pos - 2
        ),
        min(
            len(X),
            pos + 3
        )
    ):

        window_indices.add(p)


SUSPICIOUS_WINDOWS = X.iloc[
    sorted(
        window_indices
    )
].copy()


# ============================================================
# 7. 日別異常数
# ============================================================

daily = X.copy()

daily["date"] = (
    daily.index.normalize()
)


DAILY_AUDIT = (

    daily

    .groupby(
        "date"
    )

    .agg(

        bars=(
            "close",
            "size"
        ),

        min_close=(
            "close",
            "min"
        ),

        max_close=(
            "close",
            "max"
        ),

        median_close=(
            "close",
            "median"
        ),

        extreme_gaps=(
            "extreme_gap",
            "sum"
        ),

        immediate_reversals=(
            "immediate_reversal",
            "sum"
        ),

        max_abs_gap=(
            "open_gap",
            lambda s:
            s.abs().max()
        ),

        median_abs_bar_return=(
            "bar_return",
            lambda s:
            s.abs().median()
        ),
    )

    .reset_index()
)


DAILY_AUDIT[
    "daily_price_span"
] = (

    DAILY_AUDIT[
        "max_close"
    ]

    /

    DAILY_AUDIT[
        "min_close"
    ]

    -

    1
)


SUSPICIOUS_DAYS = (

    DAILY_AUDIT.loc[

        (
            DAILY_AUDIT[
                "extreme_gaps"
            ]
            >
            0
        )

        |

        (
            DAILY_AUDIT[
                "daily_price_span"
            ]
            >
            0.05
        )

    ]

    .sort_values(

        [
            "extreme_gaps",
            "daily_price_span"
        ],

        ascending=False
    )
)


# ============================================================
# 8. 年/月別異常
# ============================================================

MONTHLY = X.copy()

MONTHLY["month"] = (
    MONTHLY.index.to_period(
        "M"
    ).astype(str)
)


MONTHLY_AUDIT = (

    MONTHLY

    .groupby(
        "month"
    )

    .agg(

        bars=(
            "close",
            "size"
        ),

        extreme_gaps=(
            "extreme_gap",
            "sum"
        ),

        reversals=(
            "immediate_reversal",
            "sum"
        ),

        min_close=(
            "close",
            "min"
        ),

        max_close=(
            "close",
            "max"
        ),
    )

    .reset_index()
)


MONTHLY_AUDIT[
    "price_span"
] = (

    MONTHLY_AUDIT[
        "max_close"
    ]

    /

    MONTHLY_AUDIT[
        "min_close"
    ]

    -

    1
)


MONTHLY_SUSPICIOUS = (

    MONTHLY_AUDIT.loc[

        MONTHLY_AUDIT[
            "extreme_gaps"
        ]
        >
        0

    ]

    .sort_values(
        "extreme_gaps",
        ascending=False
    )
)


# ============================================================
# 9. Jan 2025詳細
# ============================================================

JAN2025 = X.loc[
    "2025-01-01":
    "2025-01-31 23:59:59"
].copy()


JAN2025_BAD = JAN2025.loc[

    JAN2025[
        "extreme_gap"
    ]

    |

    JAN2025[
        "immediate_reversal"
    ]

].copy()


# ============================================================
# 10. 価格帯を簡易分類
#
# 本当に2系列なら同一日に
# 大きく離れたclustersが出る
# ============================================================

def daily_cluster_summary(group):

    prices = group[
        "close"
    ].dropna()


    if len(prices) == 0:

        return pd.Series()


    q25 = prices.quantile(
        0.25
    )

    q50 = prices.quantile(
        0.50
    )

    q75 = prices.quantile(
        0.75
    )


    return pd.Series(
        {
            "q25":
                q25,

            "median":
                q50,

            "q75":
                q75,

            "min":
                prices.min(),

            "max":
                prices.max(),

            "max_min_pct":
                prices.max()
                /
                prices.min()
                -
                1,
        }
    )


CLUSTER_DAYS = (

    X.assign(
        date=X.index.normalize()
    )

    .groupby(
        "date"
    )

    .apply(
        daily_cluster_summary
    )

    .reset_index()
)


CLUSTER_DAYS = CLUSTER_DAYS.loc[

    CLUSTER_DAYS[
        "max_min_pct"
    ]
    >
    0.05

].sort_values(
    "max_min_pct",
    ascending=False
)


# ============================================================
# 11. CSV provenance候補探索
#
# 実際の元CSVがNotebook周辺にあるか探す
# ============================================================

search_roots = [
    Path("."),
    Path.home(),
]


csv_candidates = []


seen = set()


for root in search_roots:

    try:

        patterns = [
            "*usdjpy*.csv",
            "*USDJPY*.csv",
            "*usd*jpy*.csv",
        ]


        for pattern in patterns:

            for path in root.rglob(
                pattern
            ):

                try:

                    resolved = path.resolve()

                except Exception:

                    continue


                if resolved in seen:

                    continue


                seen.add(
                    resolved
                )


                try:

                    size = path.stat().st_size

                except Exception:

                    size = np.nan


                csv_candidates.append(
                    {
                        "path":
                            str(path),

                        "size_bytes":
                            size,
                    }
                )


                if len(
                    csv_candidates
                ) >= 300:

                    break


            if len(
                csv_candidates
            ) >= 300:

                break


    except Exception:

        pass


CSV_CANDIDATES = pd.DataFrame(
    csv_candidates
)


# ============================================================
# 12. 最も怪しい前後行
# ============================================================

display_cols = [

    "open",
    "high",
    "low",
    "close",

    "prev_close",

    "open_gap",
    "close_gap",

    "bar_return",
    "intrabar_range",

    "next_open",
    "next_close",
    "next_gap",

    "extreme_gap",
    "immediate_reversal",
]


# ============================================================
# 13. 結果表示
# ============================================================

def show(
    title,
    df,
    n=100
):

    print()
    print("=" * 110)
    print(title)
    print("=" * 110)

    if df is None or len(df) == 0:

        print(
            "No data"
        )

        return


    try:

        display(
            df.head(n)
        )

    except Exception:

        print(
            df.head(n).to_string()
        )


show(
    "TOP EXTREME GAPS",
    EXTREME[
        display_cols
    ].sort_values(
        "open_gap",
        key=lambda s:
        s.abs(),
        ascending=False
    ),
    100
)


show(
    "IMMEDIATE REVERSALS",
    REVERSALS[
        display_cols
    ].sort_values(
        "open_gap",
        key=lambda s:
        s.abs(),
        ascending=False
    ),
    100
)


show(
    "MOST SUSPICIOUS DAYS",
    SUSPICIOUS_DAYS,
    100
)


show(
    "SUSPICIOUS MONTHS",
    MONTHLY_SUSPICIOUS,
    100
)


show(
    "JANUARY 2025 EXTREME EVENTS",
    JAN2025_BAD[
        display_cols
    ],
    150
)


show(
    "DAYS WITH >5% INTERNAL PRICE SPAN",
    CLUSTER_DAYS,
    100
)


show(
    "POSSIBLE USDJPY CSV SOURCES",
    CSV_CANDIDATES,
    150
)


# ============================================================
# 14. Automatic diagnosis
# ============================================================

extreme_count = len(
    EXTREME
)


reversal_count = len(
    REVERSALS
)


if extreme_count:

    reversal_ratio = (
        reversal_count
        /
        extreme_count
    )

else:

    reversal_ratio = 0.0


median_extreme_bar_move = (

    EXTREME[
        "bar_return"
    ]
    .abs()
    .median()

    if len(
        EXTREME
    )

    else np.nan
)


print()
print("=" * 110)
print("SOURCE CONTAMINATION DIAGNOSIS")
print("=" * 110)

print(
    "Extreme >5% gaps:",
    extreme_count
)

print(
    "Immediate opposite reversals:",
    reversal_count
)

print(
    "Reversal ratio:",
    f"{reversal_ratio:.2%}"
)

print(
    "Median intrabar move among extreme-gap rows:",
    (
        f"{median_extreme_bar_move:.4%}"
        if np.isfinite(
            median_extreme_bar_move
        )
        else "N/A"
    )
)

print(
    "Suspicious days:",
    len(
        SUSPICIOUS_DAYS
    )
)

print(
    "Suspicious months:",
    len(
        MONTHLY_SUSPICIOUS
    )
)


# 強い判定:
# gapは巨大なのにbar内部は普通
# + 逆方向へ頻繁に戻る
if (

    extreme_count > 0

    and

    np.isfinite(
        median_extreme_bar_move
    )

    and

    median_extreme_bar_move < 0.01

    and

    reversal_ratio >= 0.20

):

    SOURCE_DIAGNOSIS = (
        "STRONG_EVIDENCE_OF_MIXED_PRICE_SERIES"
    )


elif extreme_count > 0:

    SOURCE_DIAGNOSIS = (
        "SOURCE_DATA_REQUIRES_PROVENANCE_REVIEW"
    )


else:

    SOURCE_DIAGNOSIS = (
        "NO_LARGE_CONTIGUOUS_GAPS_FOUND"
    )


print()
print(
    "Decision:",
    SOURCE_DIAGNOSIS
)


if SOURCE_DIAGNOSIS == (
    "STRONG_EVIDENCE_OF_MIXED_PRICE_SERIES"
):

    print()
    print(
        "The pattern is consistent with multiple price series "
        "being interleaved in the raw dataset."
    )

    print(
        "Do NOT delete rows by return threshold."
    )

    print(
        "Rebuild the historical dataset from the original source files "
        "before rerunning the strategy."
    )


# ============================================================
# 15. 保存
# ============================================================

EXTREME.to_csv(
    OUTPUT_DIR /
    "extreme_gaps.csv",
    index_label="timestamp"
)


REVERSALS.to_csv(
    OUTPUT_DIR /
    "immediate_reversals.csv",
    index_label="timestamp"
)


SUSPICIOUS_DAYS.to_csv(
    OUTPUT_DIR /
    "suspicious_days.csv",
    index=False
)


MONTHLY_SUSPICIOUS.to_csv(
    OUTPUT_DIR /
    "suspicious_months.csv",
    index=False
)


JAN2025_BAD.to_csv(
    OUTPUT_DIR /
    "jan_2025_events.csv",
    index_label="timestamp"
)


CLUSTER_DAYS.to_csv(
    OUTPUT_DIR /
    "cluster_days.csv",
    index=False
)


CSV_CANDIDATES.to_csv(
    OUTPUT_DIR /
    "csv_candidates.csv",
    index=False
)


SOURCE_AUDIT_EXTREME = (
    EXTREME.copy()
)

SOURCE_AUDIT_REVERSALS = (
    REVERSALS.copy()
)

SOURCE_AUDIT_DAYS = (
    SUSPICIOUS_DAYS.copy()
)

SOURCE_AUDIT_MONTHS = (
    MONTHLY_SUSPICIOUS.copy()
)

SOURCE_AUDIT_CSVS = (
    CSV_CANDIDATES.copy()
)

SOURCE_AUDIT_DECISION = (
    SOURCE_DIAGNOSIS
)


print()
print("=" * 110)
print("AUDIT COMPLETE")
print("=" * 110)

print(
    "Saved to:",
    OUTPUT_DIR
)


## 元セルindex 62
構文状態：valid


In [ ]:
# ============================================================
# SOURCE CONTAMINATION AUDIT - ERROR FIX
#
# 前の長いセルの続きとして実行するだけ
# EXTREME / REVERSALS を最新のXから作り直す
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 300)

GAP_THRESHOLD = 0.05

OUTPUT_DIR = Path("source_contamination_audit")
OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 1. Xが残っているか確認
# ============================================================

if "X" not in globals():

    raise RuntimeError(
        "前のSOURCE CONTAMINATION AUDITセルの途中結果 `X` がありません。\n"
        "Kernelを再起動した場合だけ、前の長いセルをもう一度実行してください。"
    )


X = X.copy()


# ============================================================
# 2. 必要列が無ければ安全に再作成
# ============================================================

required_base = [
    "open",
    "high",
    "low",
    "close",
]

missing = [
    c
    for c in required_base
    if c not in X.columns
]

if missing:

    raise RuntimeError(
        f"XにOHLC列が不足しています: {missing}"
    )


# ------------------------------------------------------------
# prev / next time
# ------------------------------------------------------------

time_series = pd.Series(
    X.index,
    index=X.index
)


if "prev_time" not in X.columns:

    X["prev_time"] = (
        time_series.shift(1)
    )


if "next_time" not in X.columns:

    X["next_time"] = (
        time_series.shift(-1)
    )


if "prev_close" not in X.columns:

    X["prev_close"] = (
        X["close"].shift(1)
    )


if "next_open" not in X.columns:

    X["next_open"] = (
        X["open"].shift(-1)
    )


if "next_close" not in X.columns:

    X["next_close"] = (
        X["close"].shift(-1)
    )


# ------------------------------------------------------------
# continuity
# ------------------------------------------------------------

X["contiguous_prev"] = (

    (
        time_series
        -
        X["prev_time"]
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


X["contiguous_next"] = (

    (
        X["next_time"]
        -
        time_series
    )

    ==

    pd.Timedelta(
        minutes=15
    )
)


# ============================================================
# 3. Gapを再計算
# ============================================================

X["open_gap"] = np.where(

    X["contiguous_prev"],

    X["open"]
    /
    X["prev_close"]
    -
    1,

    np.nan
)


X["close_gap"] = np.where(

    X["contiguous_prev"],

    X["close"]
    /
    X["prev_close"]
    -
    1,

    np.nan
)


X["bar_return"] = (

    X["close"]
    /
    X["open"]
    -
    1
)


X["intrabar_range"] = (

    X["high"]
    /
    X["low"]
    -
    1
)


# ============================================================
# 4. Extreme gap
# ============================================================

X["extreme_gap"] = (

    X["open_gap"].abs()
    >
    GAP_THRESHOLD
)


# 次のbarで発生するgap
X["next_gap"] = (

    X["open_gap"]
    .shift(-1)
)


# ============================================================
# 5. Immediate reversal
#
# 例:
#
# 142 → 158 (+11%)
# 158 → 142 (-10%)
#
# ============================================================

X["immediate_reversal"] = (

    X["extreme_gap"]

    &

    X["contiguous_next"]

    &

    (
        X["next_gap"].abs()
        >
        GAP_THRESHOLD
    )

    &

    (
        np.sign(
            X["open_gap"]
        )

        !=

        np.sign(
            X["next_gap"]
        )
    )
)


# ============================================================
# 6. ここで作り直す
#
# 前回のエラー原因を修正する核心部分
# ============================================================

EXTREME = X.loc[
    X["extreme_gap"]
].copy()


REVERSALS = X.loc[
    X["immediate_reversal"]
].copy()


# ============================================================
# 7. Daily Audit
# ============================================================

daily = X.copy()

daily["date"] = (
    daily.index.normalize()
)


DAILY_AUDIT = (

    daily

    .groupby(
        "date"
    )

    .agg(

        bars=(
            "close",
            "size"
        ),

        min_close=(
            "close",
            "min"
        ),

        max_close=(
            "close",
            "max"
        ),

        median_close=(
            "close",
            "median"
        ),

        extreme_gaps=(
            "extreme_gap",
            "sum"
        ),

        immediate_reversals=(
            "immediate_reversal",
            "sum"
        ),

        max_abs_gap=(
            "open_gap",
            lambda s:
            s.abs().max()
        ),
    )

    .reset_index()
)


DAILY_AUDIT[
    "daily_price_span"
] = (

    DAILY_AUDIT[
        "max_close"
    ]

    /

    DAILY_AUDIT[
        "min_close"
    ]

    -

    1
)


SUSPICIOUS_DAYS = (

    DAILY_AUDIT.loc[

        (
            DAILY_AUDIT[
                "extreme_gaps"
            ]
            >
            0
        )

        |

        (
            DAILY_AUDIT[
                "daily_price_span"
            ]
            >
            0.05
        )

    ]

    .sort_values(

        [
            "extreme_gaps",
            "daily_price_span"
        ],

        ascending=False
    )
)


# ============================================================
# 8. Monthly Audit
# ============================================================

monthly = X.copy()

monthly["month"] = (
    monthly.index
    .strftime("%Y-%m")
)


MONTHLY_AUDIT = (

    monthly

    .groupby(
        "month"
    )

    .agg(

        bars=(
            "close",
            "size"
        ),

        extreme_gaps=(
            "extreme_gap",
            "sum"
        ),

        reversals=(
            "immediate_reversal",
            "sum"
        ),

        min_close=(
            "close",
            "min"
        ),

        max_close=(
            "close",
            "max"
        ),
    )

    .reset_index()
)


MONTHLY_AUDIT[
    "price_span"
] = (

    MONTHLY_AUDIT[
        "max_close"
    ]

    /

    MONTHLY_AUDIT[
        "min_close"
    ]

    -

    1
)


MONTHLY_SUSPICIOUS = (

    MONTHLY_AUDIT.loc[

        MONTHLY_AUDIT[
            "extreme_gaps"
        ]
        >
        0

    ]

    .sort_values(
        "extreme_gaps",
        ascending=False
    )
)


# ============================================================
# 9. 2025年1月
# ============================================================

JAN2025 = X.loc[
    "2025-01-01":
    "2025-01-31 23:59:59"
].copy()


JAN2025_BAD = JAN2025.loc[

    JAN2025[
        "extreme_gap"
    ]

    |

    JAN2025[
        "immediate_reversal"
    ]

].copy()


# ============================================================
# 10. 表示関数
# ============================================================

def safe_show(
    title,
    df,
    columns=None,
    n=100
):

    print()
    print("=" * 110)
    print(title)
    print("=" * 110)

    if df is None or len(df) == 0:

        print("No data")
        return


    out = df.copy()


    if columns is not None:

        existing = [
            c
            for c in columns
            if c in out.columns
        ]

        out = out[
            existing
        ]


    try:

        display(
            out.head(n)
        )

    except Exception:

        print(
            out.head(n).to_string()
        )


display_cols = [

    "open",
    "high",
    "low",
    "close",

    "prev_close",

    "open_gap",
    "close_gap",

    "bar_return",
    "intrabar_range",

    "next_open",
    "next_close",
    "next_gap",

    "extreme_gap",
    "immediate_reversal",
]


# ============================================================
# 11. 結果表示
# ============================================================

EXTREME_SORTED = (

    EXTREME.assign(
        abs_open_gap=
        EXTREME[
            "open_gap"
        ].abs()
    )

    .sort_values(
        "abs_open_gap",
        ascending=False
    )

    .drop(
        columns="abs_open_gap"
    )
)


REVERSALS_SORTED = (

    REVERSALS.assign(
        abs_open_gap=
        REVERSALS[
            "open_gap"
        ].abs()
    )

    .sort_values(
        "abs_open_gap",
        ascending=False
    )

    .drop(
        columns="abs_open_gap"
    )
)


safe_show(
    "TOP EXTREME >5% GAPS",
    EXTREME_SORTED,
    display_cols,
    100
)


safe_show(
    "IMMEDIATE OPPOSITE >5% REVERSALS",
    REVERSALS_SORTED,
    display_cols,
    100
)


safe_show(
    "MOST SUSPICIOUS DAYS",
    SUSPICIOUS_DAYS,
    None,
    100
)


safe_show(
    "SUSPICIOUS MONTHS",
    MONTHLY_SUSPICIOUS,
    None,
    100
)


safe_show(
    "JANUARY 2025 EXTREME EVENTS",
    JAN2025_BAD,
    display_cols,
    150
)


# ============================================================
# 12. Automatic Diagnosis
# ============================================================

extreme_count = int(
    len(
        EXTREME
    )
)


reversal_count = int(
    len(
        REVERSALS
    )
)


reversal_ratio = (

    reversal_count
    /
    extreme_count

    if extreme_count > 0

    else 0.0
)


median_intrabar = (

    float(

        EXTREME[
            "bar_return"
        ]
        .abs()
        .median()
    )

    if extreme_count > 0

    else np.nan
)


median_range = (

    float(

        EXTREME[
            "intrabar_range"
        ]
        .median()
    )

    if extreme_count > 0

    else np.nan
)


print()
print("=" * 110)
print("SOURCE CONTAMINATION DIAGNOSIS")
print("=" * 110)

print(
    "Extreme >5% gaps:",
    extreme_count
)

print(
    "Immediate opposite >5% reversals:",
    reversal_count
)

print(
    "Reversal ratio:",
    f"{reversal_ratio:.2%}"
)

print(
    "Median intrabar absolute return on extreme-gap rows:",
    (
        f"{median_intrabar:.4%}"
        if np.isfinite(
            median_intrabar
        )
        else "N/A"
    )
)

print(
    "Median intrabar range on extreme-gap rows:",
    (
        f"{median_range:.4%}"
        if np.isfinite(
            median_range
        )
        else "N/A"
    )
)

print(
    "Suspicious days:",
    len(
        SUSPICIOUS_DAYS
    )
)

print(
    "Suspicious months:",
    len(
        MONTHLY_SUSPICIOUS
    )
)


# ============================================================
# 13. 判定
# ============================================================

if (

    extreme_count > 0

    and

    reversal_ratio >= 0.20

    and

    np.isfinite(
        median_intrabar
    )

    and

    median_intrabar < 0.01

):

    SOURCE_AUDIT_DECISION = (
        "STRONG_EVIDENCE_OF_MIXED_PRICE_SERIES"
    )


elif extreme_count > 0:

    SOURCE_AUDIT_DECISION = (
        "SOURCE_DATA_REQUIRES_PROVENANCE_REVIEW"
    )


else:

    SOURCE_AUDIT_DECISION = (
        "NO_LARGE_CONTIGUOUS_GAPS_FOUND"
    )


print()
print(
    "FINAL DECISION:",
    SOURCE_AUDIT_DECISION
)


if SOURCE_AUDIT_DECISION == (
    "STRONG_EVIDENCE_OF_MIXED_PRICE_SERIES"
):

    print()
    print(
        "複数の価格系列がraw data内で"
        "混在している可能性が非常に高いです。"
    )

    print(
        "return > 5% の行を単純削除してはいけません。"
    )

    print(
        "元CSV/sourceからhistorical datasetを"
        "再構築する必要があります。"
    )


# ============================================================
# 14. 保存
# ============================================================

EXTREME.to_csv(
    OUTPUT_DIR /
    "extreme_gaps_fixed.csv",
    index_label="timestamp"
)


REVERSALS.to_csv(
    OUTPUT_DIR /
    "immediate_reversals_fixed.csv",
    index_label="timestamp"
)


SUSPICIOUS_DAYS.to_csv(
    OUTPUT_DIR /
    "suspicious_days_fixed.csv",
    index=False
)


MONTHLY_SUSPICIOUS.to_csv(
    OUTPUT_DIR /
    "suspicious_months_fixed.csv",
    index=False
)


JAN2025_BAD.to_csv(
    OUTPUT_DIR /
    "jan2025_extreme_events_fixed.csv",
    index_label="timestamp"
)


SOURCE_AUDIT_EXTREME = (
    EXTREME.copy()
)

SOURCE_AUDIT_REVERSALS = (
    REVERSALS.copy()
)

SOURCE_AUDIT_DAYS = (
    SUSPICIOUS_DAYS.copy()
)

SOURCE_AUDIT_MONTHS = (
    MONTHLY_SUSPICIOUS.copy()
)


print()
print("=" * 110)
print("FIX COMPLETE")
print("=" * 110)
